<a href="https://colab.research.google.com/github/LuixCabral/Lab10---AI-topics/blob/main/Lab10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U bitsandbytes transformers accelerate

**Passo 1** - Carregamento e Qauntização

In [ ]:
from google.colab import userdata
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# PASSO 1: Ingestão Eficiente (QLoRA 4-bit)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

try:
    hf_token = userdata.get('HF_TOKEN')
except:
    hf_token = None

model_id_llama3 = 'meta-llama/Meta-Llama-3-8B'

print(f"Iniciando carregamento do {model_id_llama3}...")
try:
    tokenizer_llama3 = AutoTokenizer.from_pretrained(model_id_llama3, token=hf_token)
    tokenizer_llama3.pad_token = tokenizer_llama3.eos_token

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model_llama3 = AutoModelForCausalLM.from_pretrained(
        model_id_llama3,
        quantization_config=bnb_config,
        device_map='auto',
        token=hf_token
    )

    vram_load = torch.cuda.memory_allocated() / (1024**2)
    print(f'MÉTRICA PASSO 1: VRAM ocupada pelo modelo: {vram_load:.2f} MB')
except Exception as e:
    print(f"ERRO: Verifique se você aceitou os termos no Hugging Face. Detalhes: {e}")

**Passo 2**-Geração de String (PDF´s) e Tokenização

In [ ]:
# PASSO 2: Simulando RAG Massivo (10k a 15k tokens)
if 'tokenizer_llama3' in locals():
    base_text = "O paciente apresenta histórico de hipertensão e diabetes tipo 2. "
    # Gerando texto para atingir ~12k tokens
    simulated_context = base_text * 1200

    tokens = tokenizer_llama3(simulated_context, return_tensors="pt")

    print(f"Quantidade de tokens gerados: {tokens['input_ids'].shape[1]}")
    print("Alvo: 10.000 a 15.000 tokens.")
else:
    print("Execute o Passo 1 com sucesso primeiro.")

**Passo 3**- Gargalo do Decoder

In [ ]:
# PASSO 3: O Gargalo do Decoder (Sem Cache)
if 'model_llama3' in locals():
    model_llama3.config.use_cache = False

    # Limitando contexto para o teste de estresse (impacto visível)
    input_ids = tokens["input_ids"][:, :2048].to(model_llama3.device)

    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    print("Gerando 100 tokens SEM cache (isto será lento)...")
    with torch.no_grad():
        output = model_llama3.generate(
            input_ids,
            max_new_tokens=100,
            do_sample=True,
            pad_token_id=tokenizer_llama3.eos_token_id
        )

    total_time_step3 = time.time() - start_time
    peak_vram_step3 = torch.cuda.max_memory_allocated() / (1024**2)

    print(f"MÉTRICA PASSO 3:")
    print(f"Tempo total: {total_time_step3:.2f}s")
    print(f"Pico VRAM: {peak_vram_step3:.2f} MB")
else:
    print("Modelo não carregado no Passo 1.")

**Passo 4** - Engenharia e Otimização

In [ ]:
# PASSO 4: A Engenharia de Otimização (FlashAttention-2 + KV Cache)
if hf_token:
    print("Recarregando modelo com FlashAttention-2 (Hardware Optimization)...")
    model_llama3_opt = AutoModelForCausalLM.from_pretrained(
        model_id_llama3,
        quantization_config=bnb_config,
        device_map='auto',
        token=hf_token,
        attn_implementation="flash_attention_2"
    )

    model_llama3_opt.config.use_cache = True # Software Optimization

    input_ids = tokens["input_ids"][:, :2048].to(model_llama3_opt.device)
    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    with torch.no_grad():
        output = model_llama3_opt.generate(
            input_ids,
            max_new_tokens=100,
            do_sample=True,
            pad_token_id=tokenizer_llama3.eos_token_id
        )

    total_time_step4 = time.time() - start_time
    peak_vram_step4 = torch.cuda.max_memory_allocated() / (1024**2)

    print(f"MÉTRICA PASSO 4 (OTIMIZADO):")
    print(f"Tempo total: {total_time_step4:.2f}s")
    print(f"Pico VRAM: {peak_vram_step4:.2f} MB")
else:
    print("HF_TOKEN necessário para o Passo 4.")

### Implementação do Gemini via API removida.

In [ ]:
import time

# 1. Ativando otimizações
model_llama3.config.use_cache = True

# 2. Preparando inputs (usando o limite máximo do modelo: 8192 tokens)
input_ids = tokens["input_ids"][:, :8192].to(model_llama3.device)

torch.cuda.reset_peak_memory_stats()
start_time = time.time()

print(f"Iniciando geração OTIMIZADA com context window de {input_ids.shape[1]} tokens...")

with torch.no_grad():
    output_opt = model_llama3.generate(
        input_ids,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer_llama3.eos_token_id
    )

end_time = time.time()

# Métricas
total_time_opt = end_time - start_time
peak_vram_opt = torch.cuda.max_memory_allocated() / (1024**2)

print(f"--- RESULTADOS LLAMA 3 OTIMIZADO ---")
print(f"Tempo total de geração (100 tokens): {total_time_opt:.2f} segundos")
print(f"Pico de memória VRAM: {peak_vram_opt:.2f} MB")
print(f"Tokens por segundo: {100/total_time_opt:.2f} t/s")